# Bank of Canada policy rate, every business day since April 2009

This notebook reads my dataset [Bank of Canada Policy Rate Daily](https://www.kaggle.com/datasets/johnnykhan1/bank-of-canada-policy-rate-daily), which is the raw Bank of Canada Valet CSV for series V39079 (target for the overnight rate). It loads the observations, summarizes them, lists every change in the rate and draws the full history.

Source: Bank of Canada, Valet API, series V39079. No changes were made to the data.

In [ ]:
import glob
import pandas as pd

import matplotlib.pyplot as plt
BLUE, RED = "#011F5B", "#990000"
plt.rcParams.update({"figure.facecolor": "white", "axes.facecolor": "white", "axes.edgecolor": BLUE,
                     "axes.labelcolor": BLUE, "xtick.color": BLUE, "ytick.color": BLUE, "text.color": BLUE,
                     "axes.titleweight": "bold", "axes.spines.top": False, "axes.spines.right": False})

path = glob.glob("/kaggle/input/**/bank-of-canada-policy-rate-daily/**/csv", recursive=True)[0]
lines = open(path).read().splitlines()
start = lines.index('"OBSERVATIONS"') + 1
rate = pd.read_csv(path, skiprows=start, parse_dates=["date"]).rename(columns={"V39079": "policy_rate"})
print(path)
print(rate.shape)
rate.head()

## Summary

Row count, date range, and the low, high and latest values.

In [ ]:
print("Business days:", len(rate))
print("First day:", rate.date.min().date(), " Last day:", rate.date.max().date())
print("Missing values:", int(rate.policy_rate.isna().sum()))
print("Duplicate dates:", int(rate.date.duplicated().sum()))
rate.policy_rate.describe()

## Every rate change

A change is any day where the value differs from the day before.

In [ ]:
r = rate.copy()
r["change"] = r.policy_rate.diff()
moves = r[r.change.fillna(0) != 0][["date", "policy_rate", "change"]].reset_index(drop=True)
print("Number of changes:", len(moves))
print("Increases:", int((moves.change > 0).sum()), " Cuts:", int((moves.change < 0).sum()))
moves

## The full history

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.5))
ax.step(rate.date, rate.policy_rate, where="post", color=BLUE, lw=2)
up = moves[moves.change > 0]
down = moves[moves.change < 0]
ax.scatter(up.date, up.policy_rate, color=RED, s=18, zorder=3, label="increase")
ax.scatter(down.date, down.policy_rate, color=BLUE, s=18, zorder=3, marker="v", label="cut")
ax.set_ylabel("Target for the overnight rate (%)")
ax.set_title("Bank of Canada policy rate, business daily, 2009 to 2026")
ax.legend(frameon=False)
ax.text(0, -0.14, "Source: Bank of Canada Valet API, series V39079", transform=ax.transAxes, fontsize=8)
plt.tight_layout()
plt.show()

## Notes

The series starts on 2009-04-21, so earlier years are not here. For newer values, call https://www.bankofcanada.ca/valet/observations/V39079/csv directly. This dataset and notebook are not endorsed by the Bank of Canada.